# Tune Omni 600M for Omni Mail

Start with [setup](../README.md) and [batch labeling](README.md#label-in-batches).
The examples included here are fictional. Use your own reviewed mail to learn
whether a tuning choice helps your inbox. This notebook uses the same runtime
and input handling as Omni Mail.

## 1. Load the classifier and labels

Set `mail_csv` to your private labeled export to use your own mail.
Review the `group_id` values. Put related messages and repeated templates
in the same group. Use `yes` and `no` in the `label` column.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys
from tempfile import mkdtemp
from time import perf_counter

project = Path.cwd().resolve()
if project.name == "omni-mail-lab":
    project = project.parent
if not (project / "omni_mail").is_dir():
    raise RuntimeError("Start Jupyter from the Omni Mail checkout.")
sys.path.insert(0, str(project))

from dspy.experimental import ReAnchor
from omni_mail.classifier import LocalClassifier, decision_program
from omni_mail.lab import action_credit, examples_for, load_rows, predict, score, split_rows, tune_weights
from omni_mail.profiles import STOCK, load_profile, save_profile

mail_csv = project / "omni-mail-lab/examples/sample-labels.csv"
profile = STOCK["omni-600m"]
private_dir = Path.home() / "Library/Application Support/Omni Mail Lab"
private_dir.mkdir(parents=True, exist_ok=True, mode=0o700)
run_dir = Path(mkdtemp(prefix=f"{profile.key}-", dir=private_dir))
train_rows, validation_rows, final_rows = split_rows(load_rows(mail_csv))
print("Training, validation, final sizes:", len(train_rows), len(validation_rows), len(final_rows))

## 2. Run the baseline and inspect training errors

Keep the final split for the last cell. Review a small batch of training
cases before tuning. The score charges two points per missed action
and one per false alert.

In [ ]:
start = perf_counter()
model = LocalClassifier(profile)
print("Model load seconds:", round(perf_counter() - start, 2))
train = examples_for(model, train_rows)
validation = examples_for(model, validation_rows)
start = perf_counter()
train_predictions = predict(model.program, train)
print("Training:", score(train, train_predictions))
print("Training inference seconds:", round(perf_counter() - start, 2))
original_validation = score(validation, predict(model.program, validation))
print("Validation baseline:", original_validation)
errors = [row["id"] for row, yes in zip(train_rows, train_predictions, strict=True)
          if (row["label"] == "yes") != yes]
print("First training errors to review by local ID:", errors[:10])

## 3. Fit and audit the cutoff

ReAnchor fits on training. We keep the fitted cutoff if it lowers the
validation penalty and catches at least as many actions. A small fictional
batch is a walkthrough of the process.

In [ ]:
import dspy

selected = profile
dspy.configure_cache(enable_disk_cache=False, enable_memory_cache=True)
calibration = decision_program(model.backend, profile, cache=True)
optimizer = ReAnchor(metric=action_credit, num_threads=1)
fitted = optimizer.compile(calibration, trainset=train)
new_cutoff = fitted.fields["needs_action"]["threshold"]
anchored = score(validation, predict(fitted, validation))
print("Cutoff:", profile.cutoff, "→", new_cutoff, "validation:", anchored)
if anchored["penalty"] < original_validation["penalty"] and anchored["missed"] <= original_validation["missed"]:
    selected = replace(profile, cutoff=float(new_cutoff))
print("Selected cutoff:", selected.cutoff)

## 4. Optional WeightSpace with Codex

The writer can receive the selected training email text and labels.
Validation and final email text stay local during the writer run.
Start with a small budget. Set `writer_model` to a model ID available
in your signed-in Codex CLI, or leave it blank for the CLI default.
Enable the cell only for a batch you intend to share. The cutoff stays fixed during the edit.

In [ ]:
use_codex = False
writer_model = ""
max_passes = 3

if use_codex:
    print(f"Codex writer: {writer_model or 'CLI default'}; training examples: {len(train)}; pass budget: {max_passes}.")
    patched = tune_weights(model, decision_program(model.backend, selected), train, validation,
                           run_dir / "weightspace", writer_model, max_passes, selected.cutoff)
    if patched:
        selected = patched
model.close()

## 5. Freeze, export, and check the final emails

Save the selected profile before evaluating the final set. The reload
uses the same loader as the app. Results from the fictional sample
establish that the steps work. Evaluate your own labeled mail to judge
how well the profile works for you.

In [ ]:
exported = run_dir / "profile.json"
save_profile(exported, selected)
stock = LocalClassifier(profile)
try:
    final = examples_for(stock, final_rows)
    print("Final baseline:", score(final, predict(stock.program, final)))
finally:
    stock.close()
loaded = LocalClassifier(load_profile(exported))
try:
    print("Final reloaded profile:", score(final, predict(loaded.program, final)))
finally:
    loaded.close()
print("Load in Omni Mail with --profile", exported)